Script pour l'extraction de sous images de `Imagenet_full` qui utilise les fichiers de localisation et génèrent le dataset 'Imagenet_bbox'.

https://www.kaggle.com/c/imagenet-object-localization-challenge


In [1]:
from retinotopy import *
welcome()


-----------------------------------------------------------------------------------------
On date 2025-01-05, Running learning on host obiwan.local with device mps, pytorch==2.5.1
-----------------------------------------------------------------------------------------
Welcome on macOS-15.1.1-arm64-arm-64bit


In [2]:
args = Params()
data_set_type = 'full'
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing original images
target_data_set_type = 'bbox'
args.target_root = f'{DATAROOT}/Imagenet_{target_data_set_type}' # Directory containing cropped images
os.makedirs(args.target_root, exist_ok=True)
args.folders, args.root, args.target_root

(['val', 'train'], 'data/Imagenet_full', 'data/Imagenet_bbox')

## reading localisation metadata

In [3]:
for folder in args.folders :
    with open(args.annotations, 'r') as csv_file:
        df_data = pd.read_csv(csv_file)

In [4]:
df_data

,ImageId,PredictionString,origin_size
0,ILSVRC2012_val_00048981,n03995372 85 1 499 272,"(500, 360)"
1,ILSVRC2012_val_00037956,n03481172 131 0 499 254,"(500, 333)"
2,ILSVRC2012_val_00026161,n02108000 38 0 464 280,"(500, 334)"
3,ILSVRC2012_val_00026171,n03109150 0 14 216 299,"(225, 300)"
4,ILSVRC2012_val_00008726,n02119789 255 142 454 329 n02119789 44 21 322 ...,"(500, 357)"
...,...,...,...
49995,ILSVRC2012_val_00005961,n03388043 103 0 279 472,"(333, 500)"
49996,ILSVRC2012_val_00008801,n03089624 101 286 170 374 n03089624 236 282 30...,"(500, 375)"
49997,ILSVRC2012_val_00008176,n01518878 82 98 439 498,"(481, 500)"
49998,ILSVRC2012_val_00004764,n03874293 91 111 490 420,"(500, 500)"


In [5]:
def get_boxes(df, value):
    idx = list(df['ImageId'][df['ImageId'] == value].index)
    bboxes = []
    if idx:
        for i in range(len(df["PredictionString"][idx[0]].split(' '))//5):
            pos =(5*i)
            bboxes.append({'xmin' : int(df["PredictionString"][idx[0]].split(' ')[1 + pos]),
                           'ymin' : int(df["PredictionString"][idx[0]].split(' ')[2 + pos]),
                           'xmax' : int(df["PredictionString"][idx[0]].split(' ')[3 + (5 *i)]),
                           'ymax' : int(df["PredictionString"][idx[0]].split(' ')[4 + (5 *i)])
                                        })
    return bboxes

In [6]:
get_boxes(df_data, 'n02099849_2300')

[]

In [7]:
def clean_list(list_dir, patterns=['.DS_Store', '.ipynb_checkpoints']):
    for pattern in patterns:
        if pattern in list_dir: list_dir.remove(pattern)
    return list_dir

## cropping images

In [ ]:
from PIL import Image 

def square_box(xmin, ymin, xmax, ymax):
    temp = ((xmax-xmin)-(ymax-ymin))//2 # signed radius
    if temp > 0 :
        ymin -= temp
        ymax += temp
    else:
        xmin += temp
        xmax -= temp
    return xmin, ymin, xmax, ymax


for folder in args.folders :
    # first level
    print(f'\nFolder \"{folder}\"')
    source_folder = os.path.join(args.root, folder)
    boxes_folder = os.path.join(args.target_root, folder)
    os.makedirs(boxes_folder, exist_ok=True)

    # second level
    with open(args.annotations, 'r') as csv_file:
        df_data = pd.read_csv(csv_file)
    for i_img, img_id in enumerate(Imagenet_urls_ILSVRC_2016):
        print(f'Scraping images for id \"{img_id}\" : {labels[i_img]} ', end='')
        target_folder = os.path.join(boxes_folder, img_id)
        img_source_folder = os.path.join(source_folder, img_id)
        if not os.path.isdir(target_folder):
            os.makedirs(target_folder, exist_ok=True)
            for imgs in  clean_list(os.listdir(img_source_folder)):
                data_local = os.path.join(img_source_folder, imgs)
                objects = get_boxes(df_data, imgs.split('.')[0])

                original_image = Image.open(data_local, mode='r').convert('RGB')
                for i_obj, object in enumerate(objects): #if len(obj)  > 0 :
                    xmin = object['xmin']
                    ymin = object['ymin']
                    xmax = object['xmax']
                    ymax = object['ymax']

                    crop_image = original_image.crop(square_box(xmin, ymin, xmax, ymax))
                    no = '' if i_obj==0 else f'_{i_obj}'
                    img_name = imgs.split('.')[0] + no + '.jpg'
                    crop_image.save(os.path.join(target_folder, img_name))

        print(f' - in:  {len(clean_list(os.listdir(img_source_folder)))} / out:  {len(clean_list(os.listdir(target_folder)))}')


Folder "val"
Scraping images for id "n01440764" : tench  - in:  50 / out:  52
Scraping images for id "n01443537" : goldfish  - in:  50 / out:  162
Scraping images for id "n01484850" : great_white_shark  - in:  50 / out:  56
Scraping images for id "n01491361" : tiger_shark  - in:  50 / out:  71
Scraping images for id "n01494475" : hammerhead  - in:  50 / out:  91
Scraping images for id "n01496331" : electric_ray  - in:  50 / out:  61
Scraping images for id "n01498041" : stingray  - in:  50 / out:  80
Scraping images for id "n01514668" : cock  - in:  50 / out:  65
Scraping images for id "n01514859" : hen  - in:  50 / out:  93
Scraping images for id "n01518878" : ostrich  - in:  50 / out:  62
Scraping images for id "n01530575" : brambling  - in:  50 / out:  63
Scraping images for id "n01531178" : goldfinch  - in:  50 / out:  56
Scraping images for id "n01532829" : house_finch  - in:  50 / out:  69
Scraping images for id "n01534433" : junco  - in:  50 / out:  51
Scraping images for id "n0